# 01 — Single model run

A guided tour of the toy TB model: load the inputs, build the model, run it once with the
default parameter values and look at the outputs.

Covered here:

1. **Inputs** — parameter values, prior distributions and time-variant parameters.
2. **Build and run** — construct the `summer2` model and run it with a single parameter set.
3. **Outputs** — compartment sizes, TB burden, the care cascade and the fit to the (dummy) targets.
4. **Screening demo** — the same model with one active screening campaign added.

> All data in this repository are dummy placeholders, so none of the numbers below should be
> interpreted epidemiologically.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import tbtoy.plotting as pl
import tbtoy.runner_tools as rt
from tbtoy.config import DEFAULT_MODEL_CONFIG
from tbtoy.model import get_tb_model

pd.set_option("display.max_rows", 120)


## 1. Inputs

All inputs come from the `data/` folder:

* `parameters.csv` — one row per parameter. The `value` column gives the default (fixed) value;
  rows that also specify a `distribution` become calibration priors (see notebook 02).
* `time_variant_parameters.csv` — parameters that vary over time (here, the historical treatment
  success percentage).
* `calibration_targets.csv` — the data the model is fitted to.


In [ ]:
params, priors, tv_params = rt.get_parameters_and_priors()

print(f"{len(params)} parameters, of which {len(priors)} are calibrated")
pd.Series(params, name="value").to_frame()


In [ ]:
print("Model configuration:")
for key, value in DEFAULT_MODEL_CONFIG.items():
    print(f"  {key}: {value}")

print("\nTime-variant parameters:")
pd.DataFrame(tv_params)


## 2. Build and run the model

`get_tb_model` assembles the natural history structure, passive detection and treatment, births and
deaths, any screening campaigns, and requests all derived outputs. Running it with a dictionary of
parameter values produces both compartment sizes and derived outputs.


In [ ]:
model = get_tb_model(DEFAULT_MODEL_CONFIG, tv_params)

print(f"Simulated period: {DEFAULT_MODEL_CONFIG['start_time']} to {DEFAULT_MODEL_CONFIG['end_time']}")
print(f"Compartments ({len(model.compartments)}):")
for compartment in model.compartments:
    print(f"  {compartment}")


In [ ]:
model.run(parameters=params)

derived_outputs = model.get_derived_outputs_df()
print(f"{derived_outputs.shape[1]} derived outputs available")
derived_outputs.loc[2020:2030, ["population", "tb_incidence_per100k", "tb_notifications", "tb_mortality_per100k"]]


In [ ]:
compartment_sizes = model.get_outputs_df().loc[1850:2040]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
compartment_sizes.plot(ax=axes[0], linewidth=1.2)
axes[0].set_title("Compartment sizes", fontsize=11)
axes[0].legend(fontsize=7, ncol=2)

# excluding the (dominant) never-infected compartment makes the TB states visible
compartment_sizes.drop(columns="mtb_naive").plot(ax=axes[1], linewidth=1.2)
axes[1].set_title("Compartment sizes, excluding 'mtb_naive'", fontsize=11)
axes[1].legend(fontsize=7, ncol=2)

for ax in axes:
    ax.set_xlabel("Year")
    ax.grid(alpha=0.3)
fig.tight_layout()


In [ ]:
burden_outputs = [
    "population",
    "tb_incidence_per100k",
    "tb_prevalence_per100k",
    "tb_mortality_per100k",
    "tbi_prevalence_perc",
    "viable_tbi_prevalence_perc",
    "tst_positivity_perc",
    "perc_prev_subclinical",
    "perc_prev_infectious",
]

pl.plot_outputs(derived_outputs, burden_outputs, x_lim=(1900, 2040));


## 3. Care cascade and fit to the targets

The care cascade outputs describe how prevalent TB is detected and treated. The passive detection
rate follows a historical scale-up profile (a `tanh`-based function), and the treatment success
proportion interpolates the observed series; both are held flat into the future in this baseline run.

The `BayesianCompartmentalModel` (BCM) built by `rt.build_bcm` bundles the model with the priors and
the calibration targets, so a single parameter set can be compared with the data directly.


In [ ]:
cascade_outputs = [
    "tb_notifications",
    "tb_treatment_starts",
    "n_on_treatment",
    "case_detection_prop",
    "passive_detection_rate",
    "treatment_success_prop",
]

pl.plot_outputs(derived_outputs, cascade_outputs, x_lim=(1980, 2040));


In [ ]:
bcm = rt.build_bcm()

print("Calibration targets:", list(bcm.targets.keys()))
pl.plot_single_fit(bcm, params, x_lim=(2000, 2030));


## 4. Adding an active screening campaign

Screening is added by passing `ScreeningProgram` objects to `get_tb_model`. Each program combines a
screening tool (algorithm), a time window and a total coverage achieved over that window. Here a
single two-year CXR-triage-plus-Xpert campaign reaching 70% of the population is compared with the
baseline run above; notebook 04 compares the full set of screening algorithms.


In [ ]:
from tbtoy.interventions import CXR_XPERT, ScreeningProgram

screening_program = ScreeningProgram(
    name="demo_campaign",
    tool=CXR_XPERT,
    start_time=DEFAULT_MODEL_CONFIG["scenario_start_time"],
    end_time=DEFAULT_MODEL_CONFIG["scenario_start_time"] + 2,
    coverage_perc=70.0,
)
print(screening_program)

screening_model = get_tb_model(DEFAULT_MODEL_CONFIG, tv_params, [screening_program])
screening_model.run(parameters=params)
screening_outputs = screening_model.get_derived_outputs_df()


In [ ]:
comparison_outputs = [
    "tb_incidence_per100k",
    "tb_prevalence_per100k",
    "screening_tb_detections",
    "tb_treatment_starts",
    "n_tests",
    "cum_tb_incidence",
]
runs = {"No screening": (derived_outputs, "black"), "CXR + Xpert, 70%": (screening_outputs, "tab:red")}

fig, axes = plt.subplots(2, 3, figsize=(14, 6.5))
for ax, output in zip(axes.flatten(), comparison_outputs):
    for label, (outputs_df, colour) in runs.items():
        series = outputs_df[output].loc[2020:2040]
        ax.plot(series.index, series.values, color=colour, label=label)
    ax.set_title(pl.get_title(output), fontsize=10)
    ax.set_ylim(bottom=0.0)
    ax.grid(alpha=0.3)
axes.flatten()[0].legend(fontsize=8)
fig.tight_layout()
